

### 1. Data Preprocessing & Splitting

This step handles the sequential split and applies the `StandardScaler` (fitting only on the training set to prevent look-ahead bias).

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import torch

In [3]:
df = pd.read_csv('../golden_dataset.csv')

In [7]:
df['excercise'] = df['excercise'].map({"bench_press": 0, "arm_curl": 1})

In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Assuming your dataframe is loaded into 'df'
# Note: Ensure your target column doesn't have a leading space,
# e.g., use df.rename(columns={" exercise": "exercise"}, inplace=True)

features = ["x_accel", "y_accel", "z_accel", "x_gyro", "y_gyro", "z_gyro"]
target = "excercise"

# 1. Sequential Train-Test Split (First 80% train, last 20% test)
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

# 2. Fit scaler ONLY on TRAIN, then transform both
scaler = StandardScaler()
train_df.loc[:, features] = scaler.fit_transform(train_df[features])
test_df.loc[:, features] = scaler.transform(test_df[features])

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

Train rows: 254208 | Test rows: 63553


### 2. Sliding Window Dataset

This custom PyTorch `Dataset` takes your continuous single dataframe and chops it into overlapping windows. For the window label, it uses the most frequent class (`mode`) occurring within that specific window timeframe.

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMUDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, window_size, step_size):
        self.features = df[feature_cols].values
        self.labels = df[target_col].values
        self.window_size = window_size
        self.step_size = step_size

        self.samples = []

        # Generate sliding windows
        for start_idx in range(0, len(df) - window_size + 1, step_size):
            end_idx = start_idx + window_size
            window_features = self.features[start_idx:end_idx]
            window_labels = self.labels[start_idx:end_idx]

            # The label for the window is the most frequent activity in that window
            label = np.bincount(window_labels).argmax()

            self.samples.append((window_features, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        # x shape: (window_size, num_features)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# 3. Build Datasets
window_size = 300  # Adjust based on your sensor's sampling rate
step_size = 150     # 50% overlap

train_dataset = IMUDataset(train_df, features, target, window_size, step_size)
test_dataset  = IMUDataset(test_df, features, target, window_size, step_size)

# 4. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 3. The CNN-LSTM Model

The architecture uses a 1D Convolutional layer to extract local spatial patterns from the 6 sensor channels, followed by an LSTM to track the temporal dynamics of the exercise over the window's duration.

In [18]:
import torch.nn as nn
import torch.nn.functional as F

class CNNLSTM(nn.Module):
    def __init__(self, input_size=6, num_classes=2, hidden_size=64, num_layers=1):
        super(CNNLSTM, self).__init__()

        # CNN feature extractor
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, stride=1, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.conv3 = nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1)
        self.bn3   = nn.BatchNorm1d(128)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        # Classifier
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, features)
        x = x.permute(0, 2, 1)  # (batch, features, seq_len)

        # CNN
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))  # (batch, channels, seq_len//4)
        x = F.relu(self.bn3(self.conv3(x)))

        # Prepare for LSTM
        x = x.permute(0, 2, 1)  # (batch, seq_len_reduced, channels)

        # LSTM
        lstm_out, _ = self.lstm(x)  # (batch, seq_len_reduced, hidden_dim)
        out = lstm_out[:, -1, :]    # last hidden state

        # Classifier
        out = self.fc(out)
        return out

### 4. Training and Evaluation Loop

Finally, instantiate the model and train it using standard Cross-Entropy Loss, which handles binary classification seamlessly when `num_classes=2`.

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = CNNLSTM(input_size=len(features), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:


num_epochs = 10

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_loss = train_loss / train_total
    epoch_acc = train_correct / train_total

    # --- Evaluation Phase ---
    model.eval()
    test_correct, test_total = 0, 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = test_correct / test_total
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Test Acc: {test_acc:.4f}")

Epoch 01/10 | Train Loss: 0.4873 | Train Acc: 0.7820 | Test Acc: 0.0024
Epoch 02/10 | Train Loss: 0.3907 | Train Acc: 0.8240 | Test Acc: 0.0308
Epoch 03/10 | Train Loss: 0.4268 | Train Acc: 0.8057 | Test Acc: 0.0308
Epoch 04/10 | Train Loss: 0.3549 | Train Acc: 0.8328 | Test Acc: 0.0308
Epoch 05/10 | Train Loss: 0.3501 | Train Acc: 0.8323 | Test Acc: 0.0308
Epoch 06/10 | Train Loss: 0.3577 | Train Acc: 0.8328 | Test Acc: 0.0024
Epoch 07/10 | Train Loss: 0.3154 | Train Acc: 0.8382 | Test Acc: 0.8412
Epoch 08/10 | Train Loss: 0.3440 | Train Acc: 0.8293 | Test Acc: 0.0403
Epoch 09/10 | Train Loss: 0.3337 | Train Acc: 0.8358 | Test Acc: 0.0024
Epoch 10/10 | Train Loss: 0.3388 | Train Acc: 0.8323 | Test Acc: 0.0308


In [ ]:
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    epoch_loss = train_loss / train_total
    epoch_acc = train_correct / train_total

    # --- Evaluation Phase ---
    model.eval()
    test_correct, test_total = 0, 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)

            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()

    test_acc = test_correct / test_total
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Test Acc: {test_acc:.4f}")